In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import os
import re

In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_monorail_csv(csv_paths):
    """
    Load one or more Monorail CSV files and apply basic cleaning.
    
    Parameters:
    - csv_paths: str or list of CSV file paths
    
    Returns:
    - df: Combined DataFrame with all samples
    """
    if isinstance(csv_paths, str):
        csv_paths = [csv_paths]
    
    dfs = []
    for fp in csv_paths:
        df = pd.read_csv(fp)
        
        # # Apply filters
        # df = df[(df["Non_Standard_Braking"] == 0) & (df["BC_BadStart"]==0)].copy()
        
        # Extract source from filename
        m = re.search(r"Dati(\d+)", os.path.basename(fp))
        df["Source"] = int(m.group(1)) if m else -1
        
        # Convert "xx sec" strings to float
        for col in df.select_dtypes(include="object").columns:
            if col != "Malfunction":  # Skip Malfunction column
                try:
                    df[col] = df[col].str.replace(" sec", "", regex=False).astype(float)
                except Exception:
                    pass
        
        dfs.append(df)
    
    df_combined = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df_combined)} samples from {len(csv_paths)} file(s)")
    
    return df_combined

In [ ]:
def add_wv_bin(df):
    """
    Add WV_bin column based on WV_MeanPressure.
    
    Bins:
    - 0: WV < 2
    - 1: 2 <= WV <= 3
    - 2: WV > 3
    
    Parameters:
    - df: DataFrame with WV_MeanPressure column
    
    Returns:
    - df: DataFrame with WV_bin column added
    """
    df = df.copy()
    
    if "WV_MeanPressure" in df.columns:
        df["WV_bin"] = df["WV_MeanPressure"].apply(
            lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
        )
    else:
        df["WV_bin"] = np.nan
        print("Warning: WV_MeanPressure column not found, WV_bin set to NaN")
    
    return df

In [ ]:
def remove_label_columns(df):
    """
    Remove label columns that should not be present during inference.
    
    Parameters:
    - df: DataFrame
    
    Returns:
    - df: DataFrame with label columns removed
    """
    df = df.copy()
    
    label_cols = ["LeakageLabel", "label", "Malfunction"]
    cols_to_drop = [c for c in label_cols if c in df.columns]
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print(f"Removed label columns: {cols_to_drop}")
    
    return df

In [ ]:
def get_datetime_column(df):
    """
    Detect a column that contains Time or Datetime in the DataFrame.
    Prefers columns named 'Time' or 'Datetime'; otherwise uses first column
    whose name contains 'time' or 'date' (case-insensitive) and parses as datetime.
    """
    for name in ["Time", "Datetime", "DateTime", "Date", "Timestamp"]:
        if name in df.columns:
            return name
    for col in df.columns:
        if "time" in col.lower() or "date" in col.lower():
            try:
                sample = df[col].dropna()
                if len(sample) == 0:
                    continue
                parsed = pd.to_datetime(sample.head(20), errors="coerce")
                if parsed.notna().any():
                    return col
            except Exception:
                continue
    return None

In [ ]:
def load_and_prepare_monorail(monorail_paths):
    """
    Complete pipeline: load, clean, and prepare Monorail data for prediction.
    
    Parameters:
    - monorail_paths: str or list of CSV file paths
    - features: list of feature names used during training
    - imputer: fitted imputer from training
    - scaler: fitted scaler from training
    - wv_bin: WV_bin value to filter by (default: 1)
    
    Returns:
    - X_scaled: preprocessed data ready for prediction
    - df_filtered: filtered DataFrame with original indices
    - df_full: full DataFrame with all samples (for reference)
    """
    # Load data
    df_full = load_monorail_csv(monorail_paths)
    mapping = {
    1: "T3000",
    6: "T3000",
    27: "T3000",
    30: "T3000",
    10: "4909",
    11: "4909",
    18: "4909",
    24: "4909",
    5: "4575",
    32: "4575"
}
    df_full["WagonType"] = df_full["Source"].map(mapping)
    # Add WV_bin
    df_full = add_wv_bin(df_full)
    
    # Remove label columns
    df_full = remove_label_columns(df_full)

    return df_full

# Start - model load

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Path handling (your existing logic is correct)
# ------------------------------------------------------------------
def get_base_dir():
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

PROJECT_DIR = get_base_dir()
MODEL_DIR = PROJECT_DIR / "model"

# Find all joblib files
MODEL_PATHS = sorted(MODEL_DIR.glob("*_inference.joblib"))

print(f"Found {len(MODEL_PATHS)} model(s) in {MODEL_DIR}")


# Test data load

In [ ]:
test_data_path = [
    "TestBrakefinal_data_raw_Dati30.csv",
    "TestBrakefinal_data_raw_Dati05.csv",
    "TestBrakefinal_data_raw_Dati10.csv",
    "TestBrakefinal_data_raw_Dati11.csv",
    "TestBrakefinal_data_raw_Dati18.csv",
    "TestBrakefinal_data_raw_Dati24.csv",
]

df_test = load_and_prepare_monorail(test_data_path)

df_wv1 = df_test.loc[
    (df_test["Non_Standard_Braking"].eq(0)) &
    (df_test["BC_BadStart"].eq(0)) &
    (df_test["WV_bin"].eq(1))
].copy()

print(f"Filtered test data to {len(df_wv1)} WV_bin=1 healthy samples")


In [ ]:
all_results = {}

for model_path in MODEL_PATHS:
    model_name = model_path.stem
    print(f"\nRunning inference with: {model_name}")

    bundle = joblib.load(model_path)
    pipe = bundle["pipeline"]
    features = bundle["features"]

    # Feature alignment (critical safety check)
    X_new = df_wv1[features]

    y_pred = pipe.predict(X_new)
    proba = pipe.predict_proba(X_new)
    # Multiclass: Score = probability of the predicted class
    y_score = proba[np.arange(len(y_pred)), y_pred.astype(int)]

    # Store results in a COPY
    df_out = df_wv1.copy()
    df_out["Prediction"] = y_pred
    df_out["Score"] = y_score
    df_out["Model"] = model_name

    all_results[model_name] = df_out


In [ ]:
# Build output.csv: rows where Prediction != 0 (non-Healthy); columns: Datetime, Model, WagonType, Source, Score, PredictedClass
# Filter out rows where Datetime is before 2025
dt_col = get_datetime_column(df_wv1)
output_rows = []

for model_name, df_out in all_results.items():
    # Multiclass: any non-Healthy prediction (Prediction != 0)
    non_healthy = df_out["Prediction"].astype(int) != 0
    df_pos = df_out.loc[non_healthy].copy()
    if len(df_pos) == 0:
        continue
    if dt_col:
        df_pos["_dt"] = pd.to_datetime(df_pos[dt_col], errors="coerce")
        df_pos = df_pos.dropna(subset=["_dt"])
        df_pos = df_pos[df_pos["_dt"].dt.year >= 2025]
        if len(df_pos) == 0:
            continue
        df_pos["Datetime"] = df_pos["_dt"].astype(str)
        for _, row in df_pos.iterrows():
            output_rows.append({
                "Datetime": row["Datetime"],
                "Model": model_name,
                "WagonType": row.get("WagonType", ""),
                "Source": row.get("Source", ""),
                "Score": row["Score"],
                "PredictedClass": row["Prediction"],
            })
    else:
        for _, row in df_pos.iterrows():
            output_rows.append({
                "Datetime": None,
                "Model": model_name,
                "WagonType": row.get("WagonType", ""),
                "Source": row.get("Source", ""),
                "Score": row["Score"],
                "PredictedClass": row["Prediction"],
            })

if output_rows:
    df_output = pd.DataFrame(output_rows)
    out_path = PROJECT_DIR / "output.csv"
    df_output.to_csv(out_path, index=False)
    print(f"Wrote {len(df_output)} non-Healthy prediction row(s) (2025+) to {out_path}")
else:
    print("No non-Healthy predictions (or all filtered out); output.csv not written.")

In [ ]:
# Timetable plot of df_output: Y = Source, color = PredictedClass (fault class)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if "df_output" in globals() and len(df_output) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    df_plot = df_output.copy()
    if "Datetime" in df_plot.columns and df_plot["Datetime"].notna().any():
        df_plot["_time"] = pd.to_datetime(df_plot["Datetime"], errors="coerce")
    elif "Date" in df_plot.columns and df_plot["Date"].notna().any():
        df_plot["_time"] = pd.to_datetime(df_plot["Date"], errors="coerce")
    else:
        df_plot["_time"] = df_plot.index
    df_plot = df_plot.dropna(subset=["_time"])
    if len(df_plot) == 0:
        print("No valid dates to plot.")
    else:
        # Y-axis: Source (sorted for stable ordering)
        sources = sorted(df_plot["Source"].dropna().unique(), key=lambda x: (x == "", str(x)))
        if len(sources) == 0:
            sources = df_plot["Source"].unique()
        y_map = {s: i for i, s in enumerate(sources)}
        df_plot["_y"] = df_plot["Source"].map(y_map)
        df_plot = df_plot.dropna(subset=["_y"])
        # Color by fault class (PredictedClass): 1 = Aux, 2 = Combined
        class_labels = {0: "Healthy", 1: "Aux", 2: "Combined"}
        classes = sorted(df_plot["PredictedClass"].dropna().unique(), key=lambda x: int(x) if str(x).isdigit() else x)
        colors = plt.cm.Set1(np.linspace(0, 1, max(len(classes), 2)))
        for i, cls in enumerate(classes):
            mask = df_plot["PredictedClass"] == cls
            sub = df_plot.loc[mask]
            label = class_labels.get(int(cls) if str(cls).isdigit() else cls, str(cls))
            ax.scatter(sub["_time"], sub["_y"], c=colors[i % len(colors)], label=label, alpha=0.8, s=50)
        ax.set_yticks(range(len(sources)))
        ax.set_yticklabels([str(s) for s in sources], fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.xticks(rotation=45, ha="right")
        ax.set_xlabel("Datetime")
        ax.set_ylabel("Source")
        ax.set_title("Timetable: non-Healthy prediction events (multiclass)")
        ax.legend(loc="upper left", fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No non-Healthy events to plot. Run the output cell above first.")

## Results on all data combined

In [ ]:
summary_rows = []

for model_name, df_res in all_results.items():
    pred = df_res["Prediction"].astype(int).values
    n = len(pred)

    # Healthy-only incoming assumption:
    # - True class is always 0
    # - Any prediction != 0 is a false alarm (FP_total)
    tn = np.sum(pred == 0)
    fp_aux = np.sum(pred == 1)
    fp_combined = np.sum(pred == 2)
    fp_total = fp_aux + fp_combined

    far_total = (100 * fp_total / n) if n > 0 else np.nan
    far_aux = (100 * fp_aux / n) if n > 0 else np.nan
    far_combined = (100 * fp_combined / n) if n > 0 else np.nan

    summary_rows.append({
        "Model": model_name,
        "TotalSamples": n,
        "TN_pred0": tn,
        "FP_pred1_Aux": fp_aux,
        "FP_pred2_Combined": fp_combined,
        "FP_total": fp_total,
        "FAR_total_%": far_total,
        "FAR_aux_%": far_aux,
        "FAR_combined_%": far_combined,
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    by=["FAR_total_%", "FAR_aux_%", "FAR_combined_%"],
    ascending=[True, True, True]
).reset_index(drop=True)

summary_df


# Results for each wagon and models

In [ ]:
far_rows = []

for model_name, df_res in all_results.items():
    if "WagonType" not in df_res.columns:
        raise ValueError("Column 'WagonType' not found")

    for wagon, df_w in df_res.groupby("WagonType"):
        y_pred = df_w["Prediction"].astype(int).values
        n = len(y_pred)

        # Healthy-only ground truth assumption:
        # Confusion "row" = [pred0, pred1, pred2]
        pred0 = np.sum(y_pred == 0)   # TN
        pred1 = np.sum(y_pred == 1)   # FP to Aux leakage
        pred2 = np.sum(y_pred == 2)   # FP to Combined leakage

        fp_total = pred1 + pred2
        tn = pred0

        far_total = 100 * fp_total / n if n > 0 else np.nan
        far_aux = 100 * pred1 / n if n > 0 else np.nan
        far_combined = 100 * pred2 / n if n > 0 else np.nan

        far_rows.append({
            "Model": model_name,
            "WagonType": wagon,

            # 1-row confusion matrix components
            "Pred_Healthy_0": pred0,
            "Pred_Aux_1": pred1,
            "Pred_Combined_2": pred2,

            # compact view (optional)
            "CM_HealthyOnly_[0,1,2]": [int(pred0), int(pred1), int(pred2)],

            # FAR metrics
            "TN": int(tn),
            "FP_total": int(fp_total),
            "FP_aux": int(pred1),
            "FP_combined": int(pred2),

            "TotalSamples": int(n),
            "FAR_total_%": far_total,
            "FAR_aux_%": far_aux,
            "FAR_combined_%": far_combined,
        })

far_df = pd.DataFrame(far_rows).sort_values(
    ["WagonType", "FAR_total_%", "FAR_aux_%", "FAR_combined_%"]
).reset_index(drop=True)

far_df


In [ ]:
cm_rows = []

for model_name, df_res in all_results.items():
    pred = df_res["Prediction"].astype(int).values
    cm = np.array([[np.sum(pred == 0), np.sum(pred == 1), np.sum(pred == 2)]])  # 1×3

    cm_rows.append({
        "Model": model_name,
        "CM_HealthyOnly_[pred0,pred1,pred2]": cm.tolist()[0]
    })

cm_df = pd.DataFrame(cm_rows)
cm_df


# Plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import math
from matplotlib.colors import LinearSegmentedColormap

def short_model_name(model_key: str) -> str:
    key = model_key.lower()
    if "rf" in key:
        return "RF"
    if "dt_tuned" in key:
        return "dt_tuned"
    if "knn" in key:
        return "KNN"
    if "svm" in key:
        return "SVM"
    return model_key

light_blues = LinearSegmentedColormap.from_list(
    "light_blues",
    [(0, "#eff1f3"), (0.1, "#8cc6fc"), (1, "#95d4f6")]
)

def plot_false_alarm_grid_by_wagon_multiclass(
    all_results,
    wagon_col="WagonType",
    max_cols=4,
    labels=(0, 1, 2),
    xticklabels=("Healthy", "Auxiliary\n Leakage", "Combined\n Leakage"),
    yticklabel="True\n Healthy",
    exclude_models=None
):
    """
    For each WagonType: one figure with subplots for all models.
    Each subplot is a 1x3 "healthy-only confusion matrix":
        [Pred 0, Pred 1, Pred 2]
    assuming all samples are Healthy (y_true = 0).

    FAR_total = (Pred != 0) / total
    Breakdown: FP_aux = Pred==1, FP_comb = Pred==2
    """

    # --------------------------------------------------------------
    # Collect all wagon types across all models
    # --------------------------------------------------------------
    wagons = sorted({
        w
        for df in all_results.values()
        if wagon_col in df.columns
        for w in df[wagon_col].dropna().unique()
    })

    if len(wagons) == 0:
        raise ValueError(f"No wagons found using column '{wagon_col}'")

    exclude_models = exclude_models or []

    model_names = [
        m for m in all_results.keys()
        if short_model_name(m) not in exclude_models
    ]
    # --------------------------------------------------------------
    # One figure per wagon
    # --------------------------------------------------------------
    for w in wagons:
        n_models = len(model_names)
        ncols = min(max_cols, n_models)
        nrows = math.ceil(n_models / ncols)

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(5 * ncols, 3.5 * nrows),
            constrained_layout=True
        )
        axes = np.atleast_1d(axes).reshape(nrows, ncols)

        # ----------------------------------------------------------
        # Compute confusion stats first (for consistent color scale)
        # ----------------------------------------------------------
        vmax = 1
        stats = {}

        for model in model_names:
            df_m = all_results[model]
            df_w = df_m[df_m[wagon_col] == w]

            if df_w.empty:
                stats[model] = None
                continue

            y_pred = df_w["Prediction"].astype(int).values
            y_true = np.zeros(len(y_pred), dtype=int)

            # Multiclass labels: [0,1,2]
            cm = confusion_matrix(y_true, y_pred, labels=list(labels))

            # cm is 3x3, but y_true is only class 0 => only row 0 is non-zero
            pred0 = cm[0, 0]  # TN
            pred1 = cm[0, 1]  # FP -> Aux
            pred2 = cm[0, 2]  # FP -> Combined

            vmax = max(vmax, pred0, pred1, pred2)
            stats[model] = (pred0, pred1, pred2, len(y_pred))

        # ----------------------------------------------------------
        # Plot each model
        # ----------------------------------------------------------
        for idx, model in enumerate(model_names):
            r, c = divmod(idx, ncols)
            ax = axes[r, c]

            if stats[model] is None:
                ax.axis("off")
                ax.set_title(f"{model}\n(no samples)", fontsize=10)
                continue

            pred0, pred1, pred2, total = stats[model]
            fp_total = pred1 + pred2
            far_total = fp_total / total if total > 0 else 0.0

            im = ax.imshow(
                [[pred0, pred1, pred2]],
                cmap=light_blues,
                vmin=0,
                vmax=vmax,
                aspect="auto"
            )

            # Cell annotations
            ax.text(0, 0, f"{pred0}\n(TN)", ha="center", va="center",
                    fontsize=16, fontweight="bold")
            ax.text(1, 0, f"{pred1}\n(FP1)", ha="center", va="center",
                    fontsize=16, fontweight="bold")
            ax.text(2, 0, f"{pred2}\n(FP2)", ha="center", va="center",
                    fontsize=16, fontweight="bold")

            ax.set_xticks([0, 1, 2])
            ax.set_xticklabels(list(xticklabels))
            ax.set_yticks([0])
            ax.set_yticklabels([yticklabel])

            label = short_model_name(model)
            ax.set_title(
                f"{label}",
                # f"{label}\nFAR={far_total:.2%} (FP={fp_total}/{total})\n"
                # f"FP1={pred1}, FP2={pred2}",
                fontsize=16,
                fontweight="bold"
            )

            ax.grid(False)

        # ----------------------------------------------------------
        # Hide unused axes
        # ----------------------------------------------------------
        for j in range(n_models, nrows * ncols):
            r, c = divmod(j, ncols)
            axes[r, c].axis("off")

        # ----------------------------------------------------------
        # Figure-level annotations
        # ----------------------------------------------------------
        fig.suptitle(
            f"WagonType {w} — False Alarm Analysis",
            fontsize=20,
            fontweight="bold"
        )
        fig.supxlabel("Prediction", fontsize=22)
        # fig.supylabel("Healthy-only", fontsize=16)

        # # Shared colorbar
        # cbar = fig.colorbar(
        #     im,
        #     ax=axes.ravel().tolist(),
        #     shrink=0.85,
        #     pad=0.02
        # )
        # cbar.set_label("Sample count")

    plt.show()


In [ ]:
plt.rcParams.update({
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})

In [ ]:
plot_false_alarm_grid_by_wagon_multiclass(
    all_results,
    wagon_col="WagonType",
    max_cols=4,
    exclude_models=["dt_tuned"]
)

## RF false alarm plot by wagon

Run the next cell to define the RF-only plotting helper, then use the last cell to call it.

In [ ]:
def plot_false_alarm_rf_by_wagon_multiclass(
    all_results,
    rf_model_key=None,
    wagon_col="WagonType",
    max_cols=4,
    labels=(0, 1, 2),
    xticklabels=("Healthy", "Auxiliary\n Leakage", "Combined\n Leakage"),
    yticklabel="True\n Healthy"
):
    """
    Plot only the Random Forest healthy-only false alarm results,
    with one subplot per WagonType.
    """

    rf_candidates = [m for m in all_results.keys() if short_model_name(m) == "RF"]

    if rf_model_key is None:
        if not rf_candidates:
            raise ValueError("No RF model found in all_results")
        rf_model_key = rf_candidates[0]
    elif rf_model_key not in all_results:
        raise KeyError(f"Model '{rf_model_key}' not found in all_results")

    df_rf = all_results[rf_model_key]

    if wagon_col not in df_rf.columns:
        raise ValueError(f"Column '{wagon_col}' not found in RF results")

    wagons = sorted(df_rf[wagon_col].dropna().unique())

    if len(wagons) == 0:
        raise ValueError(f"No wagons found using column '{wagon_col}'")

    n_wagons = len(wagons)
    ncols = min(max_cols, n_wagons)
    nrows = math.ceil(n_wagons / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5 * ncols, 3.5 * nrows),
        constrained_layout=True
    )
    axes = np.atleast_1d(axes).reshape(nrows, ncols)

    vmax = 1
    stats = {}

    for w in wagons:
        df_w = df_rf[df_rf[wagon_col] == w]

        if df_w.empty:
            stats[w] = None
            continue

        y_pred = df_w["Prediction"].astype(int).values
        y_true = np.zeros(len(y_pred), dtype=int)
        cm = confusion_matrix(y_true, y_pred, labels=list(labels))

        pred0 = cm[0, 0]
        pred1 = cm[0, 1]
        pred2 = cm[0, 2]

        vmax = max(vmax, pred0, pred1, pred2)
        stats[w] = (pred0, pred1, pred2, len(y_pred))

    for idx, w in enumerate(wagons):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]

        if stats[w] is None:
            ax.axis("off")
            ax.set_title(f"WagonType {w}\n(no samples)", fontsize=10)
            continue

        pred0, pred1, pred2, total = stats[w]
        fp_total = pred1 + pred2
        far_total = fp_total / total if total > 0 else 0.0

        ax.imshow(
            [[pred0, pred1, pred2]],
            cmap=light_blues,
            vmin=0,
            vmax=vmax,
            aspect="auto"
        )

        ax.text(0, 0, f"{pred0}\n(TN)", ha="center", va="center",
                fontsize=16, fontweight="bold")
        ax.text(1, 0, f"{pred1}\n(FP1)", ha="center", va="center",
                fontsize=16, fontweight="bold")
        ax.text(2, 0, f"{pred2}\n(FP2)", ha="center", va="center",
                fontsize=16, fontweight="bold")

        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(list(xticklabels))
        ax.set_yticks([0])
        ax.set_yticklabels([yticklabel])

        ax.set_title(
            # f"WagonType {w}\nFAR={far_total:.2%} (FP={fp_total}/{total})",
            f"WagonType {w}",
            fontsize=16,
            fontweight="bold"
        )
        ax.grid(False)

    for j in range(n_wagons, nrows * ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    fig.suptitle(
        f"RF False Alarm Analysis by {wagon_col}",
        fontsize=20,
        fontweight="bold"
    )
    fig.supxlabel("Prediction", fontsize=22)
    plt.show()


In [ ]:
plot_false_alarm_rf_by_wagon_multiclass(
    {
        model_name: df[df["WagonType"].astype(str).isin(["4909", "4575"])]
        for model_name, df in all_results.items()
    },
    wagon_col="WagonType",
    max_cols=4
)
